# Chapter 10: Generate, check and audit a grounded subsection

**The question.** Can a language model write a report section that states the results correctly? Chapter 10 treats report writing as a test: try several writing instructions on practice material, freeze the best one, then apply it once to held-out material. The Chapter 10 companion in `regional_case` builds and checks a report section without any language-model call; here the drafts were written live.

**What was run live.** `claude-sonnet-5` drafted 9 report subsections from packets of checked facts: 4 instructions on 2 practice (*discovery*) packets, then the chosen instruction on 1 held-out (*reserved*) packet.

**Result.** An automatic check passed 6 of the 9 drafts, but reading them showed that 7 of the 9 need substantive revision, including the held-out draft. Checking that a draft cites the right facts is not the same as checking that what it says about them is true.

**Run.** Run 20260923T220909Z repeated the whole regional study with new optimizer seeds (74101 to 74103), a new language-model proposal and a new 200,000-customer confirmation sample, then ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. Its numbers should not be mixed with those of the `regional_case` notebooks. This notebook reads the recorded calls; rerunning it makes no new call.

## Setup and source checks

Loads this run's records and checks their SHA-256 fingerprint, so every value below comes from the unchanged run.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Automatic check and selection

The 4 instructions combine two choices: keep only the most useful measurements or include all of them, and keep only the necessary qualifications or every one. A draft is `valid` when it cites every required fact and qualification; its `cost` counts the identifiers it cites. The rule picks the instruction that is valid on both practice packets with the lowest total cost, with ties going to the lower ID. That instruction then drafts the held-out packet for seed 74103.

In [2]:
reports=read('report_results.json')
display(reports['selected'])
display(pd.DataFrame([{k:v for k,v in row.items() if k!='plan'} for row in reports['rows']]))
for row in reports['rows']:
 if row['phase']=='reserved':
  print(row.get('plan',{}).get('text',row.get('error')))
print('Automatic validity means identifier coverage only, not semantic faithfulness.')

{'id': 'w1',
 'text': 'Keep only the most useful measurements. Preserve every scope qualification. '}

,instruction,packet,phase,valid,cost,scope
0,w0,seed_74101,discovery,False,10,Identifier coverage only; text faithfulness re...
1,w0,seed_74102,discovery,True,11,Identifier coverage only; text faithfulness re...
2,w1,seed_74101,discovery,True,11,Identifier coverage only; text faithfulness re...
3,w1,seed_74102,discovery,True,11,Identifier coverage only; text faithfulness re...
4,w2,seed_74101,discovery,False,9,Identifier coverage only; text faithfulness re...
5,w2,seed_74102,discovery,False,9,Identifier coverage only; text faithfulness re...
6,w3,seed_74101,discovery,True,11,Identifier coverage only; text faithfulness re...
7,w3,seed_74102,discovery,True,11,Identifier coverage only; text faithfulness re...
8,w1,seed_74103,reserved,True,11,Identifier coverage only; text faithfulness re...


This subsection summarizes the reserved-phase confirmation metrics for seed 74103 across its two evaluated regions, reflection and continuation. For the reflection region, the Brier score was 0.0844 [region:confirmation/reflection_74103/brier], with a contrast (squared probability error) of 0.0709 [region:confirmation/reflection_74103/contrast] and a sample share of 10.59% [region:confirmation/reflection_74103/share]. For the continuation region, the Brier score was 0.0854 [region:confirmation/continuation_74103/brier], with a contrast of 0.0718 [region:confirmation/continuation_74103/contrast] and a sample share of 10.28% [region:confirmation/continuation_74103/share]. The two regions show closely comparable error and contrast magnitudes, suggesting no meaningful divergence in confirmation-stage predictive behavior between reflection and continuation subsets at this seed.

Several scope qualifications constrain interpretation of these figures. The base predictor underlying these metri

**Read the output.** **In one line: instruction `w1` was chosen, and its held-out draft passes the automatic check but contains an unsupported claim.**

**Columns.** The first output is the chosen instruction. In the table, `instruction` is w0 to w3; `packet` and `phase` say which packet and whether it is practice (discovery) or held-out (reserved); `valid` means every required identifier is cited; `cost` counts the identifiers cited; `scope` notes that this is an identifier check only. The printed text is the held-out draft.

**Notice.** `w1` and `w3` were both valid on both practice packets at equal cost (11 each); the tie went to `w1`. Six of the nine drafts passed. The held-out draft cites every identifier and repeats the qualifications, but it also says the two regions show "no meaningful divergence". The packet gives point estimates only, with no uncertainty and no threshold for what counts as meaningful, so that sentence is unsupported. The automatic check cannot catch this kind of error.

## Read the drafts

After the selection was frozen, each of the nine drafts was read against its packet and given a verdict with specific findings. The drafts themselves are unchanged. This review was done after the run and was not planned in advance.

In [3]:
audit=read('report_semantic_review.json')
display(pd.DataFrame(audit['rows']))
print(audit['scope'])
print(audit['interpretation'])

,call,verdict,findings
0,discovery_w0_seed_74101,needs_revision,[Brier score alone does not establish comparab...
1,discovery_w0_seed_74102,needs_revision,[Calls positive error contrast a predictive ed...
2,discovery_w1_seed_74101,no_substantive_error_identified,[Values are correctly rounded and qualificatio...
3,discovery_w1_seed_74102,needs_revision,[Defines contrast as control-vs-treatment erro...
4,discovery_w2_seed_74101,needs_revision,[Calls lower contrast better. This search maxi...
5,discovery_w2_seed_74102,needs_revision,[Infers comparable predictive calibration from...
6,discovery_w3_seed_74101,needs_revision,[Converts absence of variance estimates in the...
7,discovery_w3_seed_74102,no_substantive_error_identified,[Exact supplied numbers and comparison directi...
8,reserved_w1_seed_74103,needs_revision,[Claims no meaningful divergence from close po...


Post-run qualitative audit, not a preregistered or independent semantic benchmark. Raw model drafts are unchanged. Identifier-based selection was frozen before this review.
Seven of nine drafts need substantive revision. Two had no substantive error identified in this review; this is not a certification. Six of nine passed identifier coverage, including the reserved draft that made an unsupported materiality statement.


**Read the output.** **In one line: 7 of the 9 drafts need substantive revision, including the held-out one.**

**Columns.** `call`: which draft (phase, instruction and packet). `verdict`: `needs_revision` or `no_substantive_error_identified`. `findings`: what the reader found.

**Notice.** Typical errors: calling a lower contrast better although the search looks for the highest contrast, reading calibration into Brier scores alone and claiming "no meaningful divergence" without a criterion. The 2 drafts with no substantive error found are not certified correct. The automatic check and the reading measured different things, and only reading the drafts found these errors.